# Essay Writer

Steps to Build a compact version of an essay writer (AI researcher/writer agent):
1. Plan - generate a plan for the essay
2. Research Plan - Research is done based on the plan. eg: calling tavily and getting back documents
3. Generate - Follow the plan using the documents retrieved/researched to write the essay
4. Decision - Decide to either finish essay or continue 
5. If we continue, enter reflect node and generate a critique of cuurent essay 
6. Based on the critique, do another research step with tavily. Get another set of documents and append to the existing set of documents. Then, go back to generate step

#### _Create an Essay writer:_

Import from langraph and langchain and import sqlite saver as memory checkpointer:

    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated, List
    import operator
    from langgraph.checkpoint.sqlite import SqliteSaver
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

Setup in memory sqlite checkpointer:

    memory = SqliteSaver.from_conn_string(":memory:")

Create agent state to keep track of task, plan, draft, critique, content (list of documents from tavily), revision number, max revisions:

    class AgentState(TypedDict):
        task: str
        plan: str
        draft: str
        critique: str
        content: List[str]
        revision_number: int
        max_revisions: int

Define model to use:

    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

Define prompts for all the steps for the agent:

    PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
    Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
    or instructions for the sections."""

    RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
    be used when writing the following essay. Generate a list of search queries that will gather \
    any relevant information. Only generate 3 queries max."""


    WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
    Generate the best essay possible for the user's request and the initial outline. \
    If the user provides critique, respond with a revised version of your previous attempts. \
    Utilize all the information below as needed: 

    ------

    {content}"""

    REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
    Generate critique and recommendations for the user's submission. \
    Provide detailed recommendations, including requests for length, depth, style, etc."""

    RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
    be used when making any requested revisions (as outlined below). \
    Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


Ensure to get a list of strings from language model:

    from langchain_core.pydantic_v1 import BaseModel

    class Queries(BaseModel):
        queries: List[str]

Intialise tavily client:

    from tavily import TavilyClient
    import os
    tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

Define Plan node step function:

=> This will take in agent state and create list of messages with plan system prompt and human message with the task

=> Pass to the model and get back the plan and store that plan from the model into the agent state with key "plan"

    def plan_node(state: AgentState):
        messages = [
            SystemMessage(content=PLAN_PROMPT), 
            HumanMessage(content=state['task'])
        ]
        response = model.invoke(messages)
        return {"plan": response.content}

Define research plan node:

=> Takes in the plan and does the research to find resources/documents

=> Generate queries based on the task which is a list of queries

=> Pass research plan system prompt and human message with task into the model

=> Get list of documents in content state if already exists or makes an empty list

=> append each response from tavily that was received based on the queries generated into content

=> Return agent state "content" key with content 

    def research_plan_node(state: AgentState):
        queries = model.with_structured_output(Queries).invoke([
            SystemMessage(content=RESEARCH_PLAN_PROMPT),
            HumanMessage(content=state['task'])
        ])
        content = state['content'] or []
        for q in queries.queries:
            response = tavily.search(query=q, max_results=2)
            for r in response['results']:
                content.append(r['content'])
        return {"content": content}

Define generate node step function:

=> Prepare content from agent state and join them 

=> Create user message with task and plan

=> Create list of messages with user message and writer system prompt with the content formatted

=> Pass to the model and get response

=> Update draft key as response content

=> Update revision number by adding 1


    def generation_node(state: AgentState):
        content = "\n\n".join(state['content'] or [])
        user_message = HumanMessage(
            content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
        messages = [
            SystemMessage(
                content=WRITER_PROMPT.format(content=content)
            ),
            user_message
            ]
        response = model.invoke(messages)
        return {
            "draft": response.content, 
            "revision_number": state.get("revision_number", 1) + 1
        }


Define reflection node function:

=> Takes reflection prompt as system prompt and human messages as the draft

=> Return response which will be stored as critique 

    def reflection_node(state: AgentState):
        messages = [
            SystemMessage(content=REFLECTION_PROMPT), 
            HumanMessage(content=state['draft'])
        ]
        response = model.invoke(messages)
        return {"critique": response.content}

Define research critique node function:

=> Similar to the research function but instead passes research critique prompt and critique as human message

=> Get current content list and append new documents found from tavily 

    def research_critique_node(state: AgentState):
        queries = model.with_structured_output(Queries).invoke([
            SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
            HumanMessage(content=state['critique'])
        ])
        content = state['content'] or []
        for q in queries.queries:
            response = tavily.search(query=q, max_results=2)
            for r in response['results']:
                content.append(r['content'])
        return {"content": content}

Define continue function (conditional edge):


=> This is after the generation step and we decide to either finish or continue 

=> If revision number in state is greater than max  revisions, then end

=> If not, go back to reflect

    def should_continue(state):
        if state["revision_number"] > state["max_revisions"]:
            return END
        return "reflect"

Intialise Agent state for the graph:

    builder = StateGraph(AgentState)

Build graph:

    builder.add_node("planner", plan_node)
    builder.add_node("generate", generation_node)
    builder.add_node("reflect", reflection_node)
    builder.add_node("research_plan", research_plan_node)
    builder.add_node("research_critique", research_critique_node)

    builder.set_entry_point("planner")
    builder.add_conditional_edges(
        "generate", 
        should_continue, 
        {END: END, "reflect": "reflect"}
    )


    builder.add_edge("planner", "research_plan")
    builder.add_edge("research_plan", "generate")
    builder.add_edge("reflect", "research_critique")
    builder.add_edge("research_critique", "generate")

Compile graph with checkpointer:

    graph = builder.compile(checkpointer=memory)

Can visualise graph as an image:

    from IPython.display import Image

    Image(graph.get_graph().draw_png())

Call graph.steam to see all the steps happening and pass in task, max revision and revision numbers:

    thread = {"configurable": {"thread_id": "1"}}
    for s in graph.stream({
        'task': "what is the difference between langchain and langsmith",
        "max_revisions": 2,
        "revision_number": 1,
    }, thread):
        print(s)